# **Lab 4 - Design LLM Application**

<p>COMP4146/7125 Prompt Engineering for Generative AI</br>Semester 2, 2025/26, Dr. Shichao Ma, HKBU</p>

---
## Lab Overview

In this lab, we will move beyond simple chatting and build a **functioning LLM Application**. Specifically, we will build a simple "HK Hiking Planner" LLM application that gives advice based on *real-time* weather data rather than just training data.

We will engineer this application using a **Feedforward Architecture** (a linear pipeline of steps).

The lab is divided into three key parts:
- **Part 1**: **Application Engineering** (Constructing the 4-step pipeline: Retrieval → Snippetizing → Scoring → Assembly).
- **Part 2**: **Context Optimization** (Using Summarization to reduce token usage and noise).
- **Part 3**: **Evaluation** (Implementing "LLM-as-a-Judge" to automatically grade the quality of our responses).

We will use **Ollama** with the `gemma3:4b-cloud` model and the **HK Observatory Open Data API**.

## Setup

Install required packages and verify your Ollama connection.

### For Lab Computers Users
If you are using a desktop in the lab (stateless environment), you may need to reconfigure the model and authenticate:
1. **Pull the model**: Run `ollama pull gemma3:4b-cloud` in your terminal.
2. **Sign in**: Run `ollama signin` in your terminal to enable access to cloud models.

In [1]:
!pip install ollama requests

You should consider upgrading via the '/Users/shichaoma/Downloads/Local/Dev/DevEnv/bin/python3 -m pip install --upgrade pip' command.


In [2]:
import ollama
import requests
import json

# Configuration for local/cloud LLM
model_name = "gemma3:4b-cloud"

def submit(messages, temperature=0.7, max_tokens=500):
    """Simple helper to call Ollama Chat API"""
    try:
        response = ollama.chat(
            model=model_name,
            messages=messages,
            options={
                "temperature": temperature,
                "num_predict": max_tokens
            }
        )
        return response['message']['content']
    except Exception as e:
        return f"Error: {str(e)}"

print(f"✓ Configured to use model: {model_name}")

✓ Configured to use model: gemma3:4b-cloud


/Users/shichaoma/Downloads/Local/Dev/DevEnv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
# Test the Ollama setup and model connection
test_message = [{"role": "user", "content": "Hello! World"}]

response = submit(test_message)
print(f"Response: {response}")

Response: Hello there! 😊 

It's great to hear from you! Is there anything you'd like to chat about, or were you just saying hello?


---
## Part 1. Feedforward Application: HK Hiking Planner

A "Feedforward" application is a chain of operations where data flows in one direction: from input to output.

We will build the **Hiking Planner** by implementing the following 4-step pipeline:

1.  **Context Retrieval**: Fetch raw data from an external source (HK Observatory).
2.  **Snippetizing**: Break down the large raw data into smaller, discrete units (chunks/snippets).
3.  **Scoring & Prioritizing**: Evaluate each snippet's relevance to the user's specific query and select the best one(s).
4.  **Prompt Assembly**: Construct a final, focused prompt using only the high-priority snippets.

### User Request
First, let's define the input. The user wants a hiking recommendation for a specific time and difficulty.

In [4]:
# User Request contains some useful context
user_request = "I want to go for a hike in Hong Kong tomorrow. Give me advice."

print(f"👤 User Request: {user_request}")

👤 User Request: I want to go for a hike in Hong Kong tomorrow. Give me advice.


### Step 1: Context Retrieval (External Data)

LLMs are frozen in time—they don't know the weather *now*. To answer the user's question safely, we need to "ground" the model with external facts.

We will write a function to fetch the **9-day Weather Forecast** from the Hong Kong Observatory (HKO) Open Data API. This returns a JSON object containing temperature, humidity, and weather text for the upcoming days.

In [5]:
# Step 1: Context Retrieval
# For this example, we will fetch the 9-day weather forecast from the Hong Kong Observatory API.
def get_hk_weather_forecast():
    """Fetches the 9-day weather forecast from HK Observatory"""
    url = "https://data.weather.gov.hk/weatherAPI/opendata/weather.php?dataType=fnd&lang=en"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()
        else:
            return f"Error fetching weather: {response.status_code}"
    except Exception as e:
        return f"Exception: {str(e)}"

In [6]:
# Fetch the data
weather_data_raw = get_hk_weather_forecast()

# Print the raw data to understand its structure (a typical step in real-world scenarios to know what we are working with)
weather_data_raw

{'generalSituation': 'A strong northeast monsoon will affect the coast of southern China tomorrow, it will be rather cool in the morning over the region. A band of clouds will cover Guangdong in the next couple of days. The monsoon will moderate progressively and temperatures will rise midweek this week. A fresh easterly airstream is expected to affect the coast of Guangdong in the latter part of this week.',
 'weatherForecast': [{'forecastDate': '20260209',
   'week': 'Monday',
   'forecastWind': 'East force 5, occasionally up to force 7 offshore and on high ground.',
   'forecastWeather': 'Mainly cloudy and cool. One or two light rain patches in the morning and at night.',
   'forecastMaxtemp': {'value': 17, 'unit': 'C'},
   'forecastMintemp': {'value': 14, 'unit': 'C'},
   'forecastMaxrh': {'value': 90, 'unit': 'percent'},
   'forecastMinrh': {'value': 65, 'unit': 'percent'},
   'ForecastIcon': 60,
   'PSR': 'Low'},
  {'forecastDate': '20260210',
   'week': 'Tuesday',
   'forecastWi

In [7]:
# Display the structure of the raw data to understand what we are working with
print("Raw API Response keys:", weather_data_raw.keys() if isinstance(weather_data_raw, dict) else "Error")

Raw API Response keys: dict_keys(['generalSituation', 'weatherForecast', 'updateTime', 'seaTemp', 'soilTemp'])


### <b style="color:orange"> Task 1: Write your code to figure out the number of forecast days from HKO API</b>

**Number of forecast days:** 9 

In [8]:
# TODO: Write your code to figure out the number of forecast days from the API response.
# YOUR CODE HERE

print(f"Number of forecast days: {len(weather_data_raw.get('weatherForecast', []))}")

Number of forecast days: 9


In [9]:
weather_data_raw.get('weatherForecast', [])

[{'forecastDate': '20260209',
  'week': 'Monday',
  'forecastWind': 'East force 5, occasionally up to force 7 offshore and on high ground.',
  'forecastWeather': 'Mainly cloudy and cool. One or two light rain patches in the morning and at night.',
  'forecastMaxtemp': {'value': 17, 'unit': 'C'},
  'forecastMintemp': {'value': 14, 'unit': 'C'},
  'forecastMaxrh': {'value': 90, 'unit': 'percent'},
  'forecastMinrh': {'value': 65, 'unit': 'percent'},
  'ForecastIcon': 60,
  'PSR': 'Low'},
 {'forecastDate': '20260210',
  'week': 'Tuesday',
  'forecastWind': 'East force 4, force 5 at first.',
  'forecastWeather': 'Mainly cloudy. One or two light rain patches at first. Bright periods in the afternoon.',
  'forecastMaxtemp': {'value': 21, 'unit': 'C'},
  'forecastMintemp': {'value': 16, 'unit': 'C'},
  'forecastMaxrh': {'value': 90, 'unit': 'percent'},
  'forecastMinrh': {'value': 65, 'unit': 'percent'},
  'ForecastIcon': 60,
  'PSR': 'Low'},
 {'forecastDate': '20260211',
  'week': 'Wednesday

### Step 2: Snippetizing

**Why do we need this?**
The raw JSON returned by an API is often a large, deeply nested blob of data. Feeding the *entire* raw JSON to an LLM can be:
1.  **Confusing**: It may contain irrelevant keys (like internal IDs or formatting codes).
2.  **Inefficient**: It wastes tokens on structure (brackets, quotes) rather than content.

**Snippetizing** is the process of breaking this data into clean, human-readable chunks (snippets). For weather data, a logical "chunk" is **one day's forecast**. We will format each day into a clear string like:
`"Date: 20260214 | Weather: Sunny | Temp: 18-23°C"`

In [10]:
# Step 2: Snippetizing
# We will convert the list of dictionaries into a list of strings (snippets)
forecasts = weather_data_raw.get('weatherForecast', [])
snippets = []

for day in forecasts:
    # Extract key information safely using .get()
    date = day.get('forecastDate')
    week = day.get('week')
    weather = day.get('forecastWeather')
    wind = day.get('forecastWind')
    temp_max = day.get('forecastMaxtemp', {}).get('value')
    temp_min = day.get('forecastMintemp', {}).get('value')
    
    # Format date to YYYY-MM-DD for easier LLM reading
    if date and len(date) == 8:
        date_str = f"{date[:4]}-{date[4:6]}-{date[6:]}"
    else:
        date_str = date
    
    # Construct the readable string (snippet)
    snippet = f"Date: {date_str} ({week}) | Weather: {weather} | Wind: {wind} | Temp: {temp_min}-{temp_max}°C"
    snippets.append(snippet)

In [11]:
print(f"Created {len(snippets)} snippets.")
for i in range(len(snippets)):
    print(f"Example Snippet {i+1}:", snippets[i])

Created 9 snippets.
Example Snippet 1: Date: 2026-02-09 (Monday) | Weather: Mainly cloudy and cool. One or two light rain patches in the morning and at night. | Wind: East force 5, occasionally up to force 7 offshore and on high ground. | Temp: 14-17°C
Example Snippet 2: Date: 2026-02-10 (Tuesday) | Weather: Mainly cloudy. One or two light rain patches at first. Bright periods in the afternoon. | Wind: East force 4, force 5 at first. | Temp: 16-21°C
Example Snippet 3: Date: 2026-02-11 (Wednesday) | Weather: Mainly cloudy. Sunny periods during the day. | Wind: Northeast force 3. | Temp: 18-24°C
Example Snippet 4: Date: 2026-02-12 (Thursday) | Weather: Sunny intervals. | Wind: East to northeast force 4 to 5. | Temp: 17-21°C
Example Snippet 5: Date: 2026-02-13 (Friday) | Weather: Sunny periods. | Wind: East force 4, occasionally force 5 offshore at first. | Temp: 18-22°C
Example Snippet 6: Date: 2026-02-14 (Saturday) | Weather: Sunny periods .Visibility relatively low in some areas in the

### Step 3: Scoring & Prioritizing

Now that we have a list of clean snippets (one for each of the 9 days), we need to decide **which one is relevant**. 

The user asked for advice for **"tomorrow"**. We shouldn't distract the model with weather for Tuesday or next Wednesday. 

In large-scale systems, this step is often done by **Vector Search** (finding text with similar meaning). Since we have a small number of snippets, we can use an **LLM** as a "Scorer" or "Selector". We will show the LLM all the snippets and ask it to return the **Index ID** of the most relevant one.

### <b style="color:orange"> Task 2: Manually select the best matched snippet and assign it to the variable *top_snippet*</b>

Look at the snippets generated above and manually select the correct index.

In [12]:
top_snippet = None

# TODO: Assign the most relevant snippet to top_snippet based on the user_request.
# Hint: snippets is a list. Access the correct index (e.g., snippets[0]).
# YOUR CODE HERE

top_snippet = snippets[0]

print(f"Manually Selected Snippet: {top_snippet}")

Manually Selected Snippet: Date: 2026-02-09 (Monday) | Weather: Mainly cloudy and cool. One or two light rain patches in the morning and at night. | Wind: East force 5, occasionally up to force 7 offshore and on high ground. | Temp: 14-17°C


### Automated Scoring with LLM
Now, let's automate specific selection using the LLM. This makes our application flexible—if the user asks for "next Tuesday", the LLM can intelligently pick the right date without us writing complex date-parsing code.

In [13]:
# Step 3: Scoring & Prioritizing with LLM
# We will ask the LLM to identify which snippet index is most relevant to the query.
import datetime

# Get current date to help the model reason about relative dates like "tomorrow" or "this Saturday"
current_date = datetime.date.today().strftime("%Y-%m-%d (%A)")
print(f"Current Date for Context: {current_date}")

# Format the list of snippets with IDs so the LLM can reference them
snippets_text = "\n".join([f"[{i}] {s}" for i, s in enumerate(snippets)])

# Construct the selection prompt
selection_prompt = f"""
You are a retrieval system.
Current Date: {current_date}
User Query: "{user_request}"

Based on the date the user is referring to in the query.

Available Weather Documents:
{snippets_text}

Task: Return ONLY the integer index of the document that matches the date in the user query.

Response format: just the number (e.g. 0).
"""

# Submit to LLM
selected_index_str = submit([{"role": "user", "content": selection_prompt}], temperature=0).strip()

print("The index given by the LLM: ", selected_index_str)

Current Date for Context: 2026-02-08 (Sunday)
The index given by the LLM:  0


In [14]:
try:
    # Clean up response to get just the number (LLMs sometimes add periods or whitespace)
    import re
    match = re.search(r'\d+', selected_index_str)
    if match:
        selected_index = int(match.group())
        top_snippet = snippets[selected_index]
        print(f"✅ LLM Selected Index: {selected_index}")
        print(f"✅ Top Snippet: {top_snippet}")
    else:
        # If we can't find a number, we can default to the first snippet or handle it as an error.
        print("Could not parse index, using first snippet as fallback.")
        top_snippet = snippets[0]
except Exception as e:
    # If there's any error (e.g., index out of range, parsing error), we can handle it gracefully.
    print(f"Error parsing selection: {e}")
    top_snippet = snippets[0]

✅ LLM Selected Index: 0
✅ Top Snippet: Date: 2026-02-09 (Monday) | Weather: Mainly cloudy and cool. One or two light rain patches in the morning and at night. | Wind: East force 5, occasionally up to force 7 offshore and on high ground. | Temp: 14-17°C


### Step 4: Prompt Assembly & Generation

This is the final step of our pipeline. We perform **Grounding**:
We construct a prompt that combines:
1.  **System Persona**: "You are an expert Hiking Guide..."
2.  **Selected Context**: The *specific* weather snippet we chose in Step 3.
3.  **User Query**: The original question.

By feeding ONLY the relevant context (Step 3) instead of the whole database (Step 1), we ensure the model focuses on exactly the right information.

In [15]:
# Step 4: Prompt Assembly
final_prompt = f"""
You are an expert Hiking Guide in Hong Kong.

RELEVANT WEATHER CONTEXT:
{top_snippet}

USER QUERY:
{user_request}

INSTRUCTIONS:
Explain suitability based on the weather context above.
"""

print("Generating Final Response...")
final_response = submit([{"role": "user", "content": final_prompt}])

print("\n" + "="*40)
print("🤖 HK HIKING PLANNER RESPONSE")
print("="*40)
print(final_response)

Generating Final Response...

🤖 HK HIKING PLANNER RESPONSE
Okay, fantastic! Hong Kong has some brilliant hiking options, but let’s talk about tomorrow, February 9th, before we commit. 

Based on the forecast – mainly cloudy and cool with potential light rain – here’s my advice for your hike:

**Suitability Assessment:**

The weather is a little tricky, and we need to be smart about it. The east wind force 5-7 is the biggest factor. This means it will be breezy, and it *will* pick up significantly on higher ground and offshore.  That wind can make hiking quite uncomfortable, especially if it’s combined with rain. 

**Here's a breakdown of what I’d recommend:**

* **Lower Elevation Trails are Best:**  Let’s focus on trails below 400-500 meters. This will significantly reduce the impact of the wind.
* **Avoid Coastal Trails:** Because of the strong offshore wind, I'd strongly advise against any coastal hikes. The wind will be brutal and the conditions unpredictable.
* **Rain Gear is Essen

### <b style="color:orange"> Task 3: Is the current information (weather) enough to plan the hiking? What other information may be required for the application to make a concrete hiking plan? </b>

**Your answers:** 

No, the current weather information alone is not sufficient to make a concrete hiking plan. While weather dictates the safety and feasibility of a hike, a comprehensive planning application would require additional information such as:

1.  **Trail Difficulty & User Fitness Level**: Matching the user's capabilities with trail grading (easy, moderate, hard).
2.  **Trail Details**: Specific route maps, elevation profiles, duration, and distance.
3.  **User Preferences**: Desired scenery (coastal, mountain, city view), preferred start/end times, and available time.
4.  **Logistics**: Public transportation options to the start point and from the end point, or parking availability.
5.  **Equipment Lists**: Recommended gear based on the specific trail and weather conditions.
6.  **Crowd Levels**: Real-time or historical data on trail popularity to avoid crowds if desired.

---
## Part 2. Context Optimization: Summarization

In Part 1, we selected a *single* snippet. But what if we need to provide *multiple* days of context, or what if the single snippet itself is very long?

**Token Efficiency** is a critical part of Application Engineering. 
Passing raw data is expensive and slow. 
**Summarization** allows us to compress information—keeping the *signal* and removing the *noise*.

Now the user's request is shifted to finding a suitable date for hiking based on the weather forecast.

In [16]:
user_request_p2 = "What day is suitable for hiking in next few days in Hong Kong?"

print(f"👤 User Request: {user_request_p2}")

👤 User Request: What day is suitable for hiking in next few days in Hong Kong?


In [17]:
weather_data_raw

{'generalSituation': 'A strong northeast monsoon will affect the coast of southern China tomorrow, it will be rather cool in the morning over the region. A band of clouds will cover Guangdong in the next couple of days. The monsoon will moderate progressively and temperatures will rise midweek this week. A fresh easterly airstream is expected to affect the coast of Guangdong in the latter part of this week.',
 'weatherForecast': [{'forecastDate': '20260209',
   'week': 'Monday',
   'forecastWind': 'East force 5, occasionally up to force 7 offshore and on high ground.',
   'forecastWeather': 'Mainly cloudy and cool. One or two light rain patches in the morning and at night.',
   'forecastMaxtemp': {'value': 17, 'unit': 'C'},
   'forecastMintemp': {'value': 14, 'unit': 'C'},
   'forecastMaxrh': {'value': 90, 'unit': 'percent'},
   'forecastMinrh': {'value': 65, 'unit': 'percent'},
   'ForecastIcon': 60,
   'PSR': 'Low'},
  {'forecastDate': '20260210',
   'week': 'Tuesday',
   'forecastWi

In [18]:
# top_snippet

In [19]:
# Task: Create a "Summarizer" Prompt
# We ask the model to look at the huge raw JSON and compress it.
summarizer_system_msg = "You are a data assistant. Summarize the weather forecast for the next few days from the JSON provided. Output a short 1-sentence summary of the condition, temperature, and wind for each day."

raw_context_str = json.dumps(weather_data_raw)

summary_messages = [
    {"role": "system", "content": summarizer_system_msg},
    {"role": "user", "content": raw_context_str}
]

print("Summarizing weather data...")
# We use a low max_tokens to force brevity
weather_summary = submit(summary_messages)

print(f"\n📝 Weather Summary: {weather_summary}\n")

Summarizing weather data...

📝 Weather Summary: Here’s a one-sentence summary of the weather forecast for each day:

*   **Monday (Feb 9th):** It will be cool and cloudy with light rain possible, accompanied by east force 5 winds.
*   **Tuesday (Feb 10th):** Expect cloudy skies with light rain at first, transitioning to brighter afternoons and east force 4 winds.
*   **Wednesday (Feb 11th):**  The weather will be mainly cloudy with sunny periods and a northeast force 3 wind.
*   **Thursday (Feb 12th):** Sunny intervals are forecast with an east to northeast force 4 wind.
*   **Friday (Feb 13th):**  It will be sunny with east force 4 winds.
*   **Saturday (Feb 14th):**  Sunny periods are expected with relatively low visibility in the morning and at night, and a light east force 3 wind.
*   **Sunday (Feb 15th):**  Sunny periods with relatively low visibility in the morning and at night, and an east force 2 to 3 wind.
*   **Monday (Feb 16th):** Mainly cloudy with sunny intervals and a lig

In [20]:
# Compare data size
raw_len = len(raw_context_str)
sum_len = len(weather_summary)

print(f"Original Context Size: ~{raw_len} chars")
print(f"Summarized Context Size: ~{sum_len} chars")
print(f"Reduction: {((raw_len-sum_len)/raw_len)*100:.1f}% space saved!")

Original Context Size: ~4604 chars
Summarized Context Size: ~1081 chars
Reduction: 76.5% space saved!


In [21]:
# Generate a new plan using the optimized (summarized) context
optimized_prompt = f"""
You are an expert Hiking Guide.

Current Weather Condition: 
{weather_summary}

User Request: 
{user_request_p2}

Suggest a hiking trail based on this weather. Answer the question within one short paragraph.
"""

optimized_plan = submit([{"role": "user", "content": optimized_prompt}], temperature=0)
print(optimized_plan)

Okay, based on the forecast, the next few days – **Thursday (Feb 12th) and Friday (Feb 13th)** – look like the most promising for hiking in Hong Kong. Thursday offers mostly sunny intervals with a moderate east-to-northeast wind, while Friday is simply sunny with a consistent east wind. For a great hike on Friday, I’d recommend **Dragon’s Back**. It’s a relatively easy 8km trail with stunning coastal views, well-maintained paths, and it won’t be overly affected by the east wind. Remember to check trail conditions before you go and bring layers as the temperature can fluctuate!


### <b style="color:orange"> Task 4: Write your code to incorporate the raw weather information in JSON into the prompt instead of the summarized information, and compare the completions generated by the LLM. </b>

In [22]:
# TODO: Try using the raw context instead of the summary in the prompt.
# YOUR CODE HERE

raw_prompt = f"""
You are an expert Hiking Guide.

Current Weather Condition: 
{raw_context_str}

User Request: 
{user_request_p2}

Suggest a hiking trail based on this weather. Answer the question within one short paragraph.
"""

print("Generating plan using raw context...")
raw_plan = submit([{"role": "user", "content": raw_prompt}], temperature=0)
print(raw_plan)

Generating plan using raw context...
Okay, based on the forecast, Wednesday (20260211) looks like the best day for hiking. It’s expected to be mainly cloudy with sunny periods and a maximum temperature of 24°C, offering a good balance of weather conditions. For a suitable trail, I’d recommend Dragon’s Back Trail – it offers stunning coastal views and is moderately challenging, allowing you to enjoy the sunshine while being prepared for some cloud cover. Remember to check local trail conditions before you go and bring layers as the humidity will be high.


**Observation:**

1.  **Correctness**: Both the **Summarized Context** and the **Raw Context** (JSON) allow the LLM to generate a valid hiking plan. The LLM is powerful enough to parse the JSON structure directly and extract the relevant weather information.
2.  **Efficiency**: The key difference is **token usage**. Passing the full raw JSON `raw_context_str` consumes significantly more tokens than the `weather_summary`. This makes the raw approach more expensive and slower.
3.  **Trade-off**: The summarization step successfully compressed the data (removing syntax overhead like brackets and quotes) without losing the critical signal (weather conditions), proving that we can optimize context without sacrificing response quality.

---
## Part 3. Evaluation: LLM-as-a-Judge

How do we know if our Hiking Planner is actually good? 
- We can't write a unit test for "good advice".
- Manual review is slow and expensive.

The solution is **LLM-as-a-Judge**. We use a powerful LLM to evaluate the output of our application.

We will set up a competition:
1.  **Candidate A**: Our Context-Aware Planner (from Part 2).
2.  **Candidate B**: A "Blind" Model (answering without any weather context).

We will then write a **Judge Prompt** that instructs the LLM to pick the winner based on specific criteria (Specificity, Safety, Actionability).

In [23]:
# Let's create two different outputs to judge

# Output A: Grounded (The one we generated in Part 1/2 with weather data)
candidate_a = optimized_plan

# Output B: Ungrounded (Hallucinated or Generic)
# We simulate this by asking the model WITHOUT any weather context
candidate_b = submit([{"role": "user", "content": user_request_p2}])

print("--- Candidate A (Context-Aware) ---")
print(candidate_a[:500] + "...")
print("\n--- Candidate B (Generic/Blind) ---")
print(candidate_b[:500] + "...")

--- Candidate A (Context-Aware) ---
Okay, based on the forecast, the next few days – **Thursday (Feb 12th) and Friday (Feb 13th)** – look like the most promising for hiking in Hong Kong. Thursday offers mostly sunny intervals with a moderate east-to-northeast wind, while Friday is simply sunny with a consistent east wind. For a great hike on Friday, I’d recommend **Dragon’s Back**. It’s a relatively easy 8km trail with stunning coastal views, well-maintained paths, and it won’t be overly affected by the east wind. Remember to chec...

--- Candidate B (Generic/Blind) ---
Okay, let's look at the weather forecast for Hong Kong over the next few days and find the best hiking days! As of today, November 3, 2023, here's the breakdown:

**Overall Trend:** Hong Kong is experiencing a period of unsettled weather with showers and some cloud cover. However, there are pockets of sunshine and drier periods.

**Here's a day-by-day look:**

* **November 4 (Tomorrow):**  Partly cloudy with a chance of

In [ ]:
# Define the Judge's Persona and Criteria
judge_prompt = f"""
You are an impartial judge evaluating AI hiking assistants.

User Query: "{user_request_p2}"

Context: "{weather_data_raw}"

Compare these two responses:

[RESPONSE A]
{candidate_a}

[RESPONSE B]
{candidate_b}

EVALUATION CRITERIA:
1. Specificity: Does it mention actual HK locations?
2. Safety/Grounding: Does it consider the actual weather conditions (e.g. rain/heat)?
3. Helpfulness: Is the advice actionable?

Which response is better? Reply with "Response A" or "Response B" and a short reason why.
"""

print("👨‍⚖️ The LLM Judge is deciding...")
verdict = submit([{"role": "user", "content": judge_prompt}], temperature=0)

print("\n" + "="*40)
print("VERDICT")
print("="*40)
print(verdict)

👨‍⚖️ The LLM Judge is deciding...

VERDICT
Response A

Reason: Response A directly addresses the user’s request by suggesting specific days (Thursday and Friday) and even recommending a particular hiking trail (Dragon’s Back) based on the forecast and trail characteristics. It also provides helpful context about the wind and temperature. Response B, while providing a broader weather overview, is less focused on the immediate request for hiking days and offers less concrete advice. It’s also dated with the reference to “November 3, 2023,” which is irrelevant to the provided forecast.


### <b style="color:orange"> Task 5: Experiment with Noise </b>

A robust system should be able to ignore irrelevant information ("noise").
1.  Create a "Fake Context" string that contains weather info for **London** (raining, cold) instead of Hong Kong.
2.  Pass this fake context to your prompt.
3.  See if the Hiking Planner gets confused and recommends indoor activities, or if it notices the location mismatch (if you prompted it to be careful!).
4.  Write down your observation.


In [25]:
# TODO Experiment with Noise
# YOUR CODE HERE

# 1. Create a "Fake Context" string for London

fake_context = "Location: London | Weather: Heavy Rain | Temp: 8°C | Warning: Flood risk"

# 2. Pass this fake context to your prompt
# We add a specific instruction to check for location mismatch to make it "robust"
noise_test_prompt = f"""
You are an expert Hiking Guide in Hong Kong.

RELEVANT WEATHER CONTEXT:
{fake_context}

USER QUERY:
{user_request}

INSTRUCTIONS:
Explain suitability based on the weather context above.
IMPORTANT: Check if the weather context provided is for Hong Kong. 
If the weather location (e.g., London) does NOT match the hiking destination (Hong Kong), 
explicitly state: "Warning: The provided weather data is for [Location] and does not apply to Hong Kong." 
Then, provide general hiking advice for this season in Hong Kong instead.
"""

# 3. See the result
print("Generating Response with Fake Context...")
noise_response = submit([{"role": "user", "content": noise_test_prompt}], temperature=0)

print("\n" + "="*40)
print("🤖 NOISE TEST RESPONSE")
print("="*40)
print(noise_response)

Generating Response with Fake Context...

🤖 NOISE TEST RESPONSE
Okay, let’s talk about hiking in Hong Kong for tomorrow!

**Warning: The provided weather data is for London and does not apply to Hong Kong.**

Right now, London is experiencing heavy rain and a chilly 8°C – definitely not ideal for a hike! Hong Kong’s weather is going to be quite different. 

**Here’s what you can expect and what I’d recommend for hiking tomorrow:**

**Current Weather in Hong Kong (as of today - October 26, 2023):** We’re looking at generally pleasant conditions. Expect partly cloudy skies with a high of around 22°C and a low of 17°C. There’s a slight chance of showers, but nothing major. Humidity will be high, so be prepared for that.

**Hiking Advice for Hong Kong this time of year:**

* **Layering is Key:** The temperature swings can be quite noticeable, especially as the day cools down. Start with a base layer, add a fleece or light jacket, and bring a waterproof/windproof shell just in case.
* **Rai

**Observation:**

1.  **Robustness Check**: The system successfully detected the location mismatch. By adding the specific instruction ("IMPORTANT: Check if the weather context provided is for Hong Kong"), the LLM was able to reason that the provided context (London) contradicted the user's intent (hiking in Hong Kong).
2.  **Graceful Fallback**: Instead of blindly using the London rain data to recommend indoor activities in Hong Kong, the model triggered the warning and pivoted to providing general, seasonal hiking advice for Hong Kong.
3.  **Importance of Instructions**: This demonstrates that LLMs don't just "know" to check validity; they need explicit instructions to become robust against bad data (noise). Without the prompt engineering, it might have naively told the user to bring an umbrella for a sunny HK day because it saw "Heavy Rain" in the context.

### <b style="color:orange"> Submission </b>
Make sure that you **1) successfully generate all outputs in this notebook and 2) finish all the tasks**.

Submit the updated notebook with the filename ***YourName_YourStudentID.ipynb*** to the Moodle.